In [4]:
import pandas as pd
import random
import asyncio
import json
import os

from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from playwright_stealth import Stealth


In [5]:
BASE_URL = "https://forgeglobal.com"

OUTPUT_FILE = "company_links.json"

# Se il file esiste già, riprendi da lì
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r") as f:
        company_links = json.load(f)
else:
    company_links = []

company_links_set = set(company_links)


async def save_links():
    with open(OUTPUT_FILE, "w") as f:
        json.dump(
            sorted(list(company_links_set)),
            f,
            indent=2
        )


async def main():

    async with Stealth().use_async(async_playwright()) as p:

        browser = await p.chromium.launch(
            headless=False,
            slow_mo=100
        )

        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/122.0.0.0 Safari/537.36"
            ),
            viewport={"width": 1280, "height": 720}
        )

        page = await context.new_page()

        for page_num in range(1, 3):

            url = f"{BASE_URL}/search-companies/?page={page_num}"

            print(f"\nOpening {url}")

            try:

                await page.goto(
                    url,
                    timeout=60000
                )

                await page.wait_for_selector(
                    "table",
                    timeout=60000
                )

                html = await page.content()

                soup = BeautifulSoup(
                    html,
                    "lxml"
                )

                rows = soup.select(
                    "td.col-title a"
                )

                new_links = 0

                for row in rows:

                    href = row.get("href")

                    if href and "_stock/" in href:

                        full_url = (
                            BASE_URL + href
                        )

                        if full_url not in company_links_set:

                            company_links_set.add(
                                full_url
                            )

                            new_links += 1

                print(
                    f"Nuovi link trovati: {new_links}"
                )

                print(
                    f"Totale link salvati: "
                    f"{len(company_links_set)}"
                )

                # SALVATAGGIO IMMEDIATO
                await save_links()

            except Exception as e:

                print(
                    f"Errore pagina {page_num}: {e}"
                )

                print(
                    "Salvataggio finale prima di uscire..."
                )

                await save_links()

                break

            await asyncio.sleep(
                random.uniform(3, 6)
            )

        await browser.close()

    await save_links()

    print(
        f"\nCompletato. "
        f"Totale aziende: {len(company_links_set)}"
    )


await main()


Opening https://forgeglobal.com/search-companies/?page=1
Errore pagina 1: Couldn't find a tree builder with the features you requested: lxml. Do you need to install a parser library?
Salvataggio finale prima di uscire...

Completato. Totale aziende: 0


In [ ]:
# =====================================================
# CONFIG
# =====================================================

BASE_URL = "https://forgeglobal.com"
LINK_FILE = "company_links.json"

OUTPUT_COMPANIES = "aziende_principale.csv"
OUTPUT_ROUNDS = "aziende_funding_rounds.csv"

CHECKPOINT_COMPANIES = "aziende_principale_partial.csv"
CHECKPOINT_ROUNDS = "aziende_funding_rounds_partial.csv"


# =====================================================
# LOAD LINKS
# =====================================================

if os.path.exists(LINK_FILE):
    with open(LINK_FILE, "r") as f:
        company_links = json.load(f)
else:
    raise Exception("company_links.json non trovato")


# =====================================================
# HUMAN BEHAVIOR SIMULATION
# =====================================================

async def human_like_behavior(page):
    """
    Simula comportamento umano:
    - scroll progressivo
    - pause random
    - micro-movimenti
    """

    # scroll verso il basso
    for _ in range(random.randint(3, 6)):
        await page.mouse.wheel(
            0,
            random.randint(200, 700)
        )
        await asyncio.sleep(random.uniform(0.3, 1.2))

    # pausa lettura
    await asyncio.sleep(random.uniform(2, 5))

    # scroll leggero verso l'alto (comportamento umano naturale)
    if random.random() < 0.7:
        await page.mouse.wheel(
            0,
            -random.randint(100, 400)
        )
        await asyncio.sleep(random.uniform(0.5, 1.5))


# =====================================================
# PARSER (ORIGINALE TUO, NON MODIFICATO LOGICAMENTE)
# =====================================================

def parse_company_page(html_content, url):

    soup = BeautifulSoup(html_content, "lxml")

    # ---------------------------
    # COMPANY INFO
    # ---------------------------

    company_info = {
        "URL": url,
        "Company": "NA",
        "Headquarters": "NA",
        "Sector": "NA",
        "Subsector": "NA",
        "Founded": "NA",
        "Total Funding": "NA",
        "Investors": "NA"
    }

    # ---------------------------
    # COMPANY NAME
    # ---------------------------

    title = soup.find("title")

    if title:
        company_name = (
            title.text
            .replace("Stock Price", "")
            .replace("| Forge", "")
            .strip()
        )
        company_info["Company"] = company_name

    # ---------------------------
    # COMPANY DETAILS
    # ---------------------------

    details_section = soup.find("div", class_="company-details")

    if details_section:

        rows = details_section.select(".label")

        for label_div in rows:

            label = (
                label_div.get_text(" ", strip=True)
                .replace(":", "")
                .strip()
            )

            value_div = label_div.find_next_sibling("div")

            value = (
                value_div.get_text(" ", strip=True)
                if value_div else "NA"
            )

            if label == "Sector":
                company_info["Sector"] = value

            elif label == "Subsector":
                company_info["Subsector"] = value

            elif label == "Founded":
                company_info["Founded"] = value

            elif label == "Headquarters":
                company_info["Headquarters"] = value

    # ---------------------------
    # TOTAL FUNDING
    # ---------------------------

    funding_text = soup.get_text(" ", strip=True)

    if "Total funding" in funding_text:
        try:
            after = funding_text.split("Total funding")[1]
            company_info["Total Funding"] = after.split(" ")[0]
        except:
            pass

    # ---------------------------
    # INVESTORS
    # ---------------------------

    investor_names = []

    investors_section = soup.find(id="investorsSection")

    if investors_section:

        investor_divs = investors_section.find_all(
            "div",
            class_="investor-name"
        )

        for inv in investor_divs:
            name = inv.get_text(" ", strip=True)
            if name:
                investor_names.append(name)

    investor_names = list(set(investor_names))

    if investor_names:
        company_info["Investors"] = ", ".join(investor_names)

    # ---------------------------
    # FUNDING ROUNDS
    # ---------------------------

    rounds_list = []

    overview_rows = soup.select("tr.overview")

    # BIG COMPANY TEMPLATE
    if overview_rows:

        for overview in overview_rows:

            try:
                cols = overview.find_all("td", recursive=False)

                cols_clean = [
                    " ".join(c.get_text(" ", strip=True).split())
                    for c in cols
                    if c.get_text(strip=True)
                ]

                if len(cols_clean) < 5:
                    continue

                round_data = {
                    "URL": url,
                    "Funding Date": cols_clean[0],
                    "Round Name": cols_clean[1],
                    "Amount Raised": cols_clean[2],
                    "Price Per Share (Overview)": cols_clean[3],
                    "Post-Money Valuation": cols_clean[4],
                    "Key Investors (Overview)": cols_clean[5] if len(cols_clean) > 5 else "NA"
                }

                # dettaglio round
                detail_row = overview.find_next_sibling("tr")

                if detail_row and "detail" in detail_row.get("class", []):

                    labels = detail_row.select(".label")

                    for label_div in labels:

                        label = (
                            label_div.get_text(" ", strip=True)
                            .replace(":", "")
                            .strip()
                        )

                        value_div = label_div.find_next_sibling("div")

                        value = (
                            value_div.get_text(" ", strip=True)
                            if value_div else "NA"
                        )

                        round_data[label] = value

                rounds_list.append(round_data)

            except Exception as e:
                print(f"Errore parsing round: {e}")

    # SMALL COMPANY TEMPLATE
    else:

        rounds_list.append({
            "URL": url,
            "Funding Date": "NA",
            "Round Name": "NA",
            "Amount Raised": "NA",
            "Price Per Share (Overview)": "NA",
            "Post-Money Valuation": "NA",
            "Key Investors (Overview)": company_info["Investors"]
        })

    return company_info, rounds_list


# =====================================================
# MAIN SCRAPER
# =====================================================

async def main():

    all_companies = []
    all_rounds = []

    async with Stealth().use_async(async_playwright()) as p:

        browser = await p.chromium.launch(
            headless=False,
            slow_mo=80
        )

        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/122.0.0.0 Safari/537.36"
            ),
            viewport={"width": 1280, "height": 720}
        )

        page = await context.new_page()

        for i, url in enumerate(company_links):

            print(f"[{i+1}/{len(company_links)}] {url}")

            try:
                await page.goto(url, timeout=60000)

                # comportamento umano PRE load
                await human_like_behavior(page)

                # attesa struttura pagina
                try:
                    await page.wait_for_selector("tr.overview", timeout=8000)
                except:
                    await page.wait_for_selector(".company-details", timeout=8000)

                # comportamento umano POST load
                await human_like_behavior(page)

                html = await page.content()

                comp_info, comp_rounds = parse_company_page(html, url)

                all_companies.append(comp_info)
                all_rounds.extend(comp_rounds)

                # checkpoint ogni 5 aziende
                if i % 5 == 0 and i > 0:

                    print("Checkpoint salvataggio...")

                    pd.DataFrame(all_companies).to_csv(
                        CHECKPOINT_COMPANIES,
                        index=False
                    )

                    pd.DataFrame(all_rounds).to_csv(
                        CHECKPOINT_ROUNDS,
                        index=False
                    )

            except Exception as e:
                print(f"Errore su {url}: {e}")
                continue

        await browser.close()

    # =================================================
    # SALVATAGGIO FINALE
    # =================================================

    df_companies = pd.DataFrame(all_companies)
    df_rounds = pd.DataFrame(all_rounds)

    df_companies.to_csv(OUTPUT_COMPANIES, index=False)
    df_rounds.to_csv(OUTPUT_ROUNDS, index=False)

    print("\n--- COMPLETATO ---")
    print(f"Aziende: {len(df_companies)}")
    print(f"Rounds: {len(df_rounds)}")


# RUN
await main()

[1/5383] https://forgeglobal.com/0x_stock/
[2/5383] https://forgeglobal.com/100-thieves_stock/
[3/5383] https://forgeglobal.com/1047-games_stock/
[4/5383] https://forgeglobal.com/10x-genomics-inc_stock/
[5/5383] https://forgeglobal.com/11x_stock/
[6/5383] https://forgeglobal.com/128-technology_stock/
Checkpoint salvataggio...
[7/5383] https://forgeglobal.com/1366-technologies_stock/
[8/5383] https://forgeglobal.com/15five_stock/
[9/5383] https://forgeglobal.com/1kosmos_stock/
[10/5383] https://forgeglobal.com/1life-healthcare_stock/
[11/5383] https://forgeglobal.com/1more_stock/
Checkpoint salvataggio...
[12/5383] https://forgeglobal.com/1password_stock/
[13/5383] https://forgeglobal.com/1qbit_stock/
[14/5383] https://forgeglobal.com/1stdibs_stock/
[15/5383] https://forgeglobal.com/1x_stock/
[16/5383] https://forgeglobal.com/23andme_stock/
Checkpoint salvataggio...
[17/5383] https://forgeglobal.com/247-ai_stock/
[18/5383] https://forgeglobal.com/24m_stock/
[19/5383] https://forgeglobal

In [10]:
df1 = pd.read_csv("aziende_principale.csv")
df2 = pd.read_csv("aziende_principale_partial.csv")

print(len(df1))
print(len(df2))

df1 = pd.read_csv("aziende_funding_rounds.csv")
df2 = pd.read_csv("aziende_funding_rounds_partial.csv")

print(len(df1))
print(len(df2))

5370
5368
21538
21529
